In [46]:
from pathlib import Path
import json
import difflib
import re
from datetime import datetime
from pathlib import Path

# === 解析 conf 成 dict（簡易巢狀處理） ===
def parse_conf_to_dict(path):
    conf_dict = {}
    stack = [conf_dict]

    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' in line and '{' not in line:
            key, val = line.split('=', 1)
            key = key.strip()
            val = val.strip().rstrip(';')
            stack[-1][key] = val
        elif '=' in line and '{' in line:
            key = line.split('=')[0].strip()
            new_dict = {}
            stack[-1][key] = new_dict
            stack.append(new_dict)
        elif '};' in line or '}' in line:
            if len(stack) > 1:
                stack.pop()
    return conf_dict

# === 找出 conf 差異 ===
def extract_conf_diff(correct_conf, error_conf):
    d1 = parse_conf_to_dict(correct_conf)
    d2 = parse_conf_to_dict(error_conf)
    diff_keys = []

    def compare_dicts(a, b, prefix=""):
        for key in a:
            if key not in b:
                diff_keys.append(prefix + key)
            elif isinstance(a[key], dict) and isinstance(b[key], dict):
                compare_dicts(a[key], b[key], prefix + key + ".")
            elif a[key] != b[key]:
                diff_keys.append(prefix + key)
        for key in b:
            if key not in a:
                diff_keys.append(prefix + key)

    compare_dicts(d1, d2)
    return diff_keys

# === 擷取錯誤 log snippet ===
def extract_log_diff(correct_log, error_log):
    with open(correct_log, 'r', encoding='utf-8', errors='ignore') as f1, \
         open(error_log, 'r', encoding='utf-8', errors='ignore') as f2:
        lines1 = f1.readlines()
        lines2 = f2.readlines()

    diff = list(difflib.unified_diff(lines1, lines2))
    added_lines = [line[1:].strip() for line in diff if line.startswith('+') and not line.startswith('+++')]
    err_lines = [l for l in added_lines if any(e in l.lower() for e in ['error', 'fail', 'assert', 'syntax', 'invalid'])]
    return err_lines[:3]



def cu_conf_to_json(conf_path, output_json_path):
    with open(conf_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    result = []
    assign_pattern = re.compile(r'^\s*([A-Za-z0-9_]+)\s*=\s*(.+?);\s*(#.*)?$')

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("#") or not stripped:
            continue  # Skip comment or empty lines

        match = assign_pattern.match(stripped)
        if match:
            key = match.group(1)
            value = match.group(2).strip()
            full_content = f"{key} = {value};"
            result.append({
                "label": key,
                "content": full_content
            })

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"✅ JSON saved to: {output_json_path}")
def du_conf_to_json(conf_path, output_json_path):
    results = []

    with open(conf_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    for line in lines:
        # 移除註解與前後空白
        line = line.strip()
        if not line or line.startswith("#") or line.startswith("//"):
            continue

        # 正規化抓取 label 與 content（只抓最外層參數）
        match = re.match(r'^([a-zA-Z0-9_]+)\s*=\s*.+;', line)
        if match:
            label = match.group(1)
            results.append({
                "label": label,
                "content": line
            })

    # 儲存為 JSON
    with open(output_json_path, 'w', encoding='utf-8') as outfile:
        json.dump(results, outfile, indent=2)
    print(f"✅ JSON saved to {output_json_path}")



# NVDIA API (LLM seting)

In [47]:
from pathlib import Path
from openai import OpenAI

# === 呼叫 NVIDIA LLM API ===
def call_nvidia_llm(prompt: str) -> str:
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key="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"
    )

    response = client.chat.completions.create(
        model="meta/llama3-70b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        top_p=0.7,
        max_tokens=1024,
        stream=False
    )

    return response.choices[0].message.content

In [48]:
baseline_dir = "/home/aiml/johnson/auto_gen_debug_yaml/success_data"
fail_dir = "/home/aiml/johnson/auto_gen_debug_yaml/fail_data"
debug_yaml_dir = "/home/aiml/johnson/auto_gen_debug_yaml/debug_yaml_dir"

test_cu_index = "10_cu_gnb_dlOffsetToCarrier_mismatch"
test_du_index = "10_du_gnb_dlOffsetToCarrier_mismatch"


# === 設定檔案路徑 ===
baseline_cu_conf = Path(f"{baseline_dir}/conf/0_cu_gnb_success.conf")
baseline_du_conf = Path(f"{baseline_dir}/conf/0_du_gnb_success.conf")
baseline_cu_log = Path(f"{baseline_dir}/log/0_cu_gnb_success.log")
baseline_du_log = Path(f"{baseline_dir}/log/0_du_gnb_success.log")

# === CU/DU 路徑 ===
error_cu_conf = Path(f"{fail_dir}/conf/{test_cu_index}.conf")
error_du_conf = Path(f"{fail_dir}/conf/{test_du_index}.conf")
error_cu_log = Path(f"{fail_dir}/log/{test_cu_index}.log")
error_du_log = Path(f"{fail_dir}/log/{test_du_index}.log")



output_yaml = Path("debug_generated.yaml")
cases = []


# === CU 差異 ===
print("=== [CU Config Diff] ===")
cu_conf_diff = extract_conf_diff(baseline_cu_conf, error_cu_conf)
for k in cu_conf_diff:
    print(f"- {k}")

print("\n=== [CU Log Diff] ===")
cu_log_diff = extract_log_diff(baseline_cu_log, error_cu_log)
for line in cu_log_diff:
    print(f">>> {line}")

# === DU 差異 ===
print("\n=== [DU Config Diff] ===")
du_conf_diff = extract_conf_diff(baseline_du_conf, error_du_conf)
for k in du_conf_diff:
    print(f"- {k}")

print("\n=== [DU Log Diff] ===")
du_log_diff = extract_log_diff(baseline_du_log, error_du_log)
for line in du_log_diff:
    print(f">>> {line}")

print("--------------------------")


cu_conf_diff_text = "\n".join(f"- {k}" for k in cu_conf_diff)
cu_log_diff_text = "\n".join(f"- {l}" for l in cu_log_diff[:5]) 
du_conf_diff_text = "\n".join(f"- {k}" for k in du_conf_diff)
du_log_diff_text = "\n".join(f"- {l}" for l in du_log_diff[:5]) 


=== [CU Config Diff] ===

=== [CU Log Diff] ===

=== [DU Config Diff] ===
- dl_offstToCarrier

=== [DU Log Diff] ===
--------------------------


In [49]:
prompt = f"""
You are a OAI 5G/O-RAN debugging assistant working on gNB configuration issues.

Below is a configuration file that caused a failure, along with the corresponding error log.

Your task:
- Analyze what caused the failure.
- Identify the key configuration parameters involved.
- Extract the relevant log messages.
- Generate a `debug.yaml` entry in the format below.

In the `notes` field, provide a **detailed explanation using 5G NR / O-RAN terminology**:
- Specify which layer is affected (e.g., PHY, MAC, RRC)
- Explain what failure behavior is expected
- If possible, connect to relevant 3GPP/O-RAN specifications or architecture
- Clarify how the misconfiguration breaks OAI's initialization logic

[CONFIG DIFFERENCE]
CU:
{cu_conf_diff_text or "No difference detected."}

DU:
{du_conf_diff_text or "No difference detected."}

[ERROR LOG]
CU:
{cu_log_diff_text or "No error log from CU."}

DU:
{du_log_diff_text or "No error log from DU."}

Please output in this format:


Please output the following YAML structure:

- stage: <cu_init / du_init / f1 / NGAP / cell search / random access / syntax error>
  type: <CU / DU>
  symptom: "<short explanation of failure>"
  log_snippet:
    - "..."
    - "..."
  related_config:
    - "<parameter1>"
  notes: |
    <Provide a technical explanation here using 5G/O-RAN context>
"""

In [50]:
llm_output = call_nvidia_llm(prompt)
print(llm_output)

# 儲存到 debug_yaml_dir，以 timestamp 命名
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = Path(debug_yaml_dir)
output_path.mkdir(parents=True, exist_ok=True)  # 確保資料夾存在

output_file = output_path / f"debug_{timestamp}.yaml"

with open(output_file, "w", encoding="utf-8") as f:
    f.write(llm_output)

Here is the analysis of the configuration file and error log:

- stage: du_init
  type: DU
  symptom: "DU initialization failure due to invalid dl_offstToCarrier configuration"
  log_snippet:
    - "No error log from DU." (Note: Although there is no explicit error log, the absence of logs indicates a failure during DU initialization)
  related_config:
    - "dl_offstToCarrier"
  notes: |
    The failure occurs during the DU initialization stage, specifically related to the `dl_offstToCarrier` configuration parameter. This parameter is crucial for downlink (DL) frequency offset calculation in the DU.

    In the O-RAN architecture, the DU is responsible for handling the physical layer (PHY) and medium access control (MAC) layers. The `dl_offstToCarrier` parameter affects the PHY layer, as it determines the frequency offset for downlink transmissions.

    A misconfigured `dl_offstToCarrier` value can lead to incorrect frequency offset calculations, causing the DU initialization to fail.